# Advanced Fruit Single Shot Detection (SSD) ModelThis notebook implements a state-of-the-art object detection model using Transfer Learning to detect and localize fruits with high precision.**Key Features:**- Transfer Learning with MobileNetV2 backbone- Advanced loss functions (Focal Loss + Smooth L1 Loss)- Proper data augmentation for object detection- Comprehensive evaluation and visualization- Model checkpointing and saving**Author:** Computer Vision Engineer  **Dataset:** Fruit Images for Object Detection (Kaggle)

## 1. Setup & ImportsInstall and import all necessary libraries.

In [ ]:
# Install required packages!pip install -q tensorflow opencv-python-headless matplotlib scikit-learn# Core importsimport osimport cv2import numpy as npimport matplotlib.pyplot as pltimport matplotlib.patches as patchesimport randomimport xml.etree.ElementTree as ETfrom sklearn.model_selection import train_test_splitimport warningswarnings.filterwarnings('ignore')# TensorFlow and Keras importsimport tensorflow as tffrom tensorflow import kerasfrom tensorflow.keras import layers, models, optimizersfrom tensorflow.keras.applications import MobileNetV2from tensorflow.keras.preprocessing.image import img_to_array, load_imgfrom tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpointimport tensorflow.keras.backend as K# Google Colab specificfrom google.colab import userdataprint(f"TensorFlow Version: {tf.__version__}")print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")# Set random seeds for reproducibilitySEED = 42np.random.seed(SEED)tf.random.set_seed(SEED)random.seed(SEED)

## 2. Configuration & HyperparametersDefine all configuration parameters.

In [ ]:
# Dataset ConfigurationKAGGLE_DATASET = "mbkinaci/fruit-images-for-object-detection"BASE_DIR = "/content/fruit_data"MODEL_TO_SAVE = "fruit_detector_v1.keras"# Model ConfigurationIMG_WIDTH = 224IMG_HEIGHT = 224GRID_SIZE = 7NUM_CLASSES = 3# Class MappingCLASS_MAP = {"apple": 0, "banana": 1, "orange": 2}INV_CLASS_MAP = {0: "apple", 1: "banana", 2: "orange"}CLASS_COLORS = {0: 'red', 1: 'yellow', 2: 'orange'}# Training HyperparametersBATCH_SIZE = 16EPOCHS = 50LEARNING_RATE = 1e-4PATIENCE = 10# Loss weightsLAMBDA_COORD = 5.0LAMBDA_NOOBJ = 0.5print("Configuration loaded successfully!")print(f"Image Size: {IMG_WIDTH}x{IMG_HEIGHT}")print(f"Grid Size: {GRID_SIZE}x{GRID_SIZE}")print(f"Classes: {list(CLASS_MAP.keys())}")

## 3. Dataset DownloadDownload the fruit detection dataset from Kaggle.

In [ ]:
# Configure Kaggle APIos.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')# Download and extract datasetif not os.path.exists(BASE_DIR):    print("Downloading dataset...")    !kaggle datasets download -d {KAGGLE_DATASET}    !unzip -q fruit-images-for-object-detection.zip -d {BASE_DIR}    print(f"Dataset downloaded to {BASE_DIR}")else:    print("Dataset already exists.")# Verify structureprint("\nDataset structure:")for folder in os.listdir(BASE_DIR):    folder_path = os.path.join(BASE_DIR, folder)    if os.path.isdir(folder_path):        print(f"  {folder}/")